In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Copyright 2017 The TensorFlow Authors All Rights Reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# =============================================================================

from __future__ import absolute_import, division, print_function

import logging
import os
import time

import tensorflow as tf

import migration.bounds.model_with_elbo as RNNlayer
from migration.config import TrainingSettings
from migration.models import vrnn
import migration.datasets as datasets


# get batch and model
def create_dataset_and_model(config, shuffle, repeat):

    inputs, targets, mmsis, time_starts, time_ends, lengths, mean =  datasets.create_AIS_dataset('../../data/ct_2017010203_10_20/ct_2017010203_10_20_train.pkl', 
                    '../../data/ct_2017010203_10_20/mean.pkl',
                    32, # batch size
                    99999, # not used lol
                    300,
                    300, 
                    30,
                    72, 
                    shuffle=False,
                    repeat=False)
    # Convert the mean of the training set to logit space so it can be used to
    # initialize the bias of the generative distribution.
    generative_bias_init = -tf.math.log(1. / tf.clip_by_value(mean, 0.0001, 0.9999) - 1)
    generative_distribution_class = vrnn.ConditionalBernoulliDistribution
    model = vrnn.create_vrnn(inputs.get_shape().as_list()[2],
                             config.latent_size,
                             generative_distribution_class,
                             generative_bias_init=generative_bias_init,
                             raw_sigma_bias=0.5)
    return inputs, targets, mmsis, time_starts, time_ends, lengths, model




def run_train(config):

    if config.random_seed: tf.random.set_seed(config.random_seed)

    inputs, targets, _, _, _, lengths, model = create_dataset_and_model(config,
                                                               shuffle=True,
                                                               repeat=True)
    optimizer = tf.keras.optimizers.Adam(learning_rate=config.learning_rate)
    
    @tf.function
    def train_step(x,y):
        with tf.GradientTape() as tape:
            bound = RNNlayer.call(model, (x, y),
                                    lengths,
                                    num_samples=1)
            

            # Compute lower bounds on the log likelihood.
            bound = tf.reduce_mean(input_tensor=bound / tf.cast(lengths, dtype=tf.float32))
            loss = -bound
        grads = tape.gradient(loss, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))
        return loss
    for epoch in range(5):
        loss_value = train_step(inputs, targets)
        print(loss_value)


In [12]:
print(config.trainingset_path)
fh = logging.FileHandler(os.path.join(config.logdir,config.log_filename+".log"))
# get TF logger
logger = logging.getLogger('tensorflow')
logger.addHandler(fh)
run_train(config)


./data/ct_2017010203_10_20/ct_2017010203_10_20_train.pkl
tf.Tensor(21.003883, shape=(), dtype=float32)
tf.Tensor(20.90087, shape=(), dtype=float32)
tf.Tensor(20.836676, shape=(), dtype=float32)
tf.Tensor(20.82341, shape=(), dtype=float32)
tf.Tensor(20.808657, shape=(), dtype=float32)
